# Chapter 62 — Digital Twins and Adaptive Structures

*A structure that reports its own strain can also model itself — this notebook builds the estimator that keeps that model honest as the structure changes.*

**Companion notebook to Chapter 62 — Digital Twins and Adaptive Structures.**

A digital twin is a computational model of a physical structure that updates continuously as new measurement data arrives. The twin is not a static simulation — it is a living representation that tracks the current state of the actual structure, not the design intent. When the structure changes — through fatigue, corrosion, thermal expansion, or load redistribution — the twin changes with it, because its parameters are continuously estimated from the measurement stream.

The concept connects directly to strain measurement: gages on the physical structure provide the observations that drive the twin's parameter estimates. A cantilever beam with a strain gage at its fixed end and a known load at the tip gives sufficient information to estimate the beam's bending stiffness *EI* — not the handbook value, but the actual stiffness of that specific beam under its current condition. As *EI* changes because a crack has formed, or because the beam has been inadvertently overloaded, the strain measurements record that change and the twin's estimates follow it.

The *Kalman filter* is the optimal tool for this tracking problem when parameters evolve according to a known noise model. It provides both the current estimate (what is *EI* right now?) and the current uncertainty (how confident are we?). The *innovation sequence* — the difference between what the measurement says and what the twin predicts — is the primary diagnostic: a persistent non-zero innovation indicates model error; a sudden spike signals a structural event worth investigating.

**This notebook builds the twin in three stages:**
1. **Static calibration** — twin model (E = 200 GPa assumed) vs. measured strain, identifying the actual E = 193 GPa by residual analysis.
2. **Recursive parameter estimation** — running-mean convergence of the stiffness estimate, with the $1/\sqrt{k}$ standard-error decay verified against theory.
3. **Kalman filter tracking** — following an 8% stiffness reduction at load cycle 1000 (fatigue-crack initiation), with the detection lag measured in cycles.

In [ ]:
# !pip install numpy matplotlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Shared plot styling
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

# Where the chapter figures are written (do not change these paths — the book
# includes these exact files)
OUT_DIR = Path("../src/media/ch62")
OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 — Static Calibration

The starting point for any digital twin is a physics model. Here the structure is a steel cantilever beam instrumented with a strain gage at the fixed end. The twin uses the standard bending formula for the surface strain at the wall under a tip load:

$$\varepsilon = \frac{6 F L}{b h^2 E}$$

In words: the strain the gage reads grows in proportion to the applied load $F$ and the beam length $L$, and falls with the beam's stiffness — a stiffer material (larger $E$) or a deeper section (larger $h$) deforms less for the same load.

| Symbol | Quantity | Value |
|--------|----------|-------|
| F | Applied load | 0–5 kN |
| L | Beam length | 500 mm |
| b | Beam width | 25 mm |
| h | Beam depth | 50 mm |
| E | Young's modulus | assumed 200 GPa |

The twin assumes nominal steel (E = 200 GPa). The actual beam, as revealed by measurement, has E = 193 GPa — a plausible 3.5% deviation from the handbook value due to heat treatment. The residual plot on the right exposes the systematic under-prediction and quantifies the model error, and the calibration step recovers the true modulus from the measured strains alone.

In [ ]:
# ── 1. Static Calibration ─────────────────────────────────────────────────────
# Cantilever beam, strain gage at the fixed end:  ε = 6·F·L / (b·h²·E)
# The twin starts from the handbook modulus; measurement reveals the real one.

# Beam geometry (realistic 500 × 25 × 50 mm steel bar)
L, b, h = 0.500, 0.025, 0.050          # m
alpha   = 6 * L / (b * h**2)           # geometry factor 6L/(bh²)  [m⁻²]

# Material moduli
E_model  = 200e9   # Pa — nominal handbook value the twin assumes
E_actual = 193e9   # Pa — true modulus (≈3.5% softer, heat-treatment variation)

MEAS_NOISE_UE = 15.0   # µε — strain-gage measurement noise (1σ)


def beam_strain_ue(force_N, modulus_Pa):
    """Fixed-end surface strain in µε for the cantilever.

    Implements ε = αF/E scaled to microstrain, where α = 6L/(bh²) is the
    geometry factor defined above. This forward model is the twin's physics.
    """
    return alpha * force_N / modulus_Pa * 1e6


np.random.seed(1)
F = np.linspace(500, 5000, 50)         # N — applied-load sweep

eps_model    = beam_strain_ue(F, E_model)
eps_measured = beam_strain_ue(F, E_actual) + np.random.normal(0, MEAS_NOISE_UE, len(F))

# Best-fit E from measurements (ratio-of-means estimator through the origin)
E_fit = alpha * F.mean() / ((eps_measured / 1e6).mean())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Digital Twin — Stage 1: Static Calibration', fontsize=12)

ax = axes[0]
ax.plot(F/1000, eps_model, 'steelblue', lw=2, ls='--',
        label=f'Twin model  (E = {E_model/1e9:.0f} GPa)')
ax.scatter(F/1000, eps_measured, s=25, alpha=0.7, color='tomato', label='Measured')
ax.plot(F/1000, beam_strain_ue(F, E_fit), 'seagreen', lw=2,
        label=f'Calibrated  (E = {E_fit/1e9:.1f} GPa)')
ax.set_xlabel('Applied force (kN)'); ax.set_ylabel('Strain (µε)')
ax.set_title('Model vs measurement')
ax.legend(fontsize=9); ax.grid(True, linestyle=':', alpha=0.4)

ax = axes[1]
residual = eps_measured - eps_model
ax.scatter(F/1000, residual, s=25, color='tomato', alpha=0.7, label='Residual')
ax.axhline(residual.mean(), color='steelblue', lw=2, ls='--',
           label=f'Mean bias = {residual.mean():.1f} µε')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Applied force (kN)'); ax.set_ylabel('Residual (µε)')
ax.set_title('Systematic model error\n(positive bias → model under-predicts strain)')
ax.legend(fontsize=9); ax.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig62_nb1_digital_twin_calibration.png', bbox_inches='tight', dpi=150)
plt.show()

# Quantify the model error before and after calibration
rms_before = np.sqrt(np.mean((eps_measured - eps_model)**2))
rms_after  = np.sqrt(np.mean((eps_measured - beam_strain_ue(F, E_fit))**2))
print(f"Assumed E:      {E_model/1e9:.0f} GPa")
print(f"Calibrated E:   {E_fit/1e9:.1f} GPa  (true: {E_actual/1e9:.0f} GPa)")
print(f"Model error:    {100*(E_model-E_actual)/E_actual:.1f}%  →  "
      f"mean strain under-prediction {residual.mean():.1f} µε")
print(f"RMS residual:   {rms_before:.1f} µε before  →  {rms_after:.1f} µε after calibration")

---
## 2 — Recursive Parameter Estimation

A single batch of measurements gives one calibrated E. But a digital twin is most useful when it continuously refines its estimate as new measurements arrive — each reading tightening the uncertainty, converging toward the true material property.

Recursive least squares (RLS) does this without storing all previous data. At each step it updates the estimate and its variance using only the new measurement and the current state:

$$\hat{E}_{k} = \hat{E}_{k-1} + K_k \bigl(\varepsilon_k^{\text{meas}} - \hat{\varepsilon}_k\bigr)$$

In words: the new stiffness estimate is the previous one, nudged by a gain $K_k$ times the *surprise* — how far the latest measured strain fell from what the current estimate predicted. As confidence grows the gain shrinks, so late measurements move the estimate less than early ones.

This is mathematically equivalent to fitting all data seen so far, but computable with O(1) memory — exactly what an embedded digital twin requires. The demonstration below uses the simplest sufficient statistic, a running mean of the per-reading stiffness, whose standard error decays as $1/\sqrt{k}$.

In [ ]:
# ── 2. Recursive Parameter Estimation ────────────────────────────────────────
# Each measurement gives an independent E estimate by inverting the beam equation:
#   ε = α·F / E  →  E_k = α·F_k / ε_k
# The running mean of E_k converges to E_actual as k grows.
# Standard error shrinks as σ/√k — each new measurement tightens the estimate.

np.random.seed(7)
n_obs = 100
F_rls   = np.random.uniform(1000, 5000, n_obs)                                  # N
eps_rls = beam_strain_ue(F_rls, E_actual) + np.random.normal(0, MEAS_NOISE_UE, n_obs)  # µε

# Per-measurement E estimate (invert beam equation, fully consistent units)
E_per = alpha * F_rls / (eps_rls * 1e-6) / 1e9   # GPa per reading

# Running mean and standard error
k_arr  = np.arange(1, n_obs + 1)
E_run  = np.cumsum(E_per) / k_arr                 # running mean (GPa)
E_se   = np.array([
    E_per[:k].std(ddof=1) / np.sqrt(k) if k > 1 else np.abs(E_per[0] - E_actual/1e9)
    for k in k_arr
])                                                 # running standard error (GPa)

# Theoretical 1/√k decay (anchored to the SE at k=5)
anchor = E_se[4] * np.sqrt(5)
E_se_theory = anchor / np.sqrt(k_arr)

obs_idx = k_arr

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Digital Twin — Stage 2: Recursive Stiffness Estimation', fontsize=12)

ax = axes[0]
ax.scatter(obs_idx, E_per, s=12, alpha=0.3, color='gray', label='Single-reading E')
ax.plot(obs_idx, E_run,  color='steelblue', lw=2.0, label='Running mean')
ax.fill_between(obs_idx,
                E_run - 2*E_se,
                E_run + 2*E_se,
                alpha=0.25, color='steelblue', label='±2 SE')
ax.axhline(E_actual/1e9, color='tomato', lw=1.5, ls='--',
           label=f'True E = {E_actual/1e9:.0f} GPa')
ax.axhline(E_model/1e9, color='gray', lw=1.0, ls=':',
           label=f'Initial guess = {E_model/1e9:.0f} GPa')
ax.set_xlabel('Measurement number')
ax.set_ylabel('Estimated E (GPa)')
ax.set_title('Running mean converges to true stiffness')
ax.legend(fontsize=9); ax.grid(True, linestyle=':', alpha=0.4)

ax = axes[1]
ax.plot(obs_idx, E_se,        color='steelblue', lw=2,   label='Observed SE')
ax.plot(obs_idx, E_se_theory, color='tomato',    lw=1.5,
        ls='--', label='Theoretical σ/√k')
ax.set_xlabel('Measurement number')
ax.set_ylabel('Standard error in E (GPa)')
ax.set_title('Uncertainty shrinks as 1/√k\n(each new measurement adds information)')
ax.legend(fontsize=9); ax.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig62_nb1_recursive_stiffness.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"After  5 measurements:  E = {E_run[4]:.2f} ± {2*E_se[4]:.2f} GPa")
print(f"After 20 measurements:  E = {E_run[19]:.2f} ± {2*E_se[19]:.2f} GPa")
print(f"After {n_obs} measurements:  E = {E_run[-1]:.2f} ± {2*E_se[-1]:.2f} GPa  "
      f"(true: {E_actual/1e9:.0f} GPa)")

---
## 3 — Kalman Filter: Tracking a Changing Structure

Recursive least squares assumes the parameter being estimated is constant. A real structure is not constant — stiffness degrades as cracks propagate, corrosion removes section, or fasteners loosen. A digital twin that stops learning once it converges will gradually fall behind reality.

The Kalman filter extends RLS by adding a **process noise** term $Q$ that keeps the estimate alive: even after convergence, the filter remains alert for change. The tradeoff is between tracking speed (large $Q$ → follows change quickly but noisier estimate) and smoothness (small $Q$ → stable but slow to respond).

**Scenario:** the beam is loaded with 2000 cycles of varying force. True stiffness is 193 GPa for the first 1000 cycles. At cycle 1001, a fatigue crack initiates and E drops abruptly to 178 GPa — an 8% reduction. The twin must detect and track this change from the strain record alone, with no direct knowledge of the damage event.

In [ ]:
# ── 3. Kalman Filter: Tracking Changing Stiffness ────────────────────────────
# State = compliance C = 1/E, so the measurement model ε = α·F·C is *linear* in
# the state. A random-walk process model (process noise Q) keeps the estimate
# alive after convergence, so it can follow a sudden stiffness loss.

def kalman_scalar_update(state, cov, meas, H, Q, R):
    """One scalar Kalman step for a random-walk state.

    The prediction step is a random walk (state unchanged, variance inflated by
    the process noise Q); the update step corrects the prediction with a single
    scalar measurement `meas` related to the state by ``meas ≈ H · state``.

    Parameters
    ----------
    state, cov : float   prior state estimate and its variance
    meas       : float   scalar measurement
    H          : float   observation gain (meas ≈ H · state)
    Q, R       : float   process- and measurement-noise variances

    Returns
    -------
    (state_post, cov_post, innovation) : tuple of float
        posterior state, posterior variance, and the innovation (residual).
    """
    # Predict (random walk)
    state_pred = state
    cov_pred   = cov + Q
    # Update
    S          = H**2 * cov_pred + R          # innovation variance
    K          = cov_pred * H / S             # Kalman gain
    innovation = meas - H * state_pred        # measurement minus prediction
    state_post = state_pred + K * innovation
    cov_post   = (1 - K * H) * cov_pred
    return state_post, cov_post, innovation


np.random.seed(42)
n_cycles   = 2000
damage_at  = 1000

E_pre  = 193e9   # Pa — healthy
E_post = 178e9   # Pa — damaged (8% stiffness loss, fatigue crack)

E_true = np.where(np.arange(n_cycles) < damage_at, E_pre, E_post)
C_true = 1.0 / E_true

# Varying load: sinusoidal with noise
F_cyc = 3000 + 1200*np.sin(np.arange(n_cycles)*0.03) + np.random.normal(0, 150, n_cycles)
F_cyc = np.maximum(F_cyc, 500)

MEAS_NOISE_STRAIN = 20e-6   # strain (20 µε) measurement noise (1σ), in SI units
eps_cyc = alpha * F_cyc * C_true + np.random.normal(0, MEAS_NOISE_STRAIN, n_cycles)

# Kalman filter tuning (state = compliance C = 1/E)
C_kf  = 1.0 / E_model         # initial guess: the handbook modulus
P_kf  = (0.05 * C_kf)**2      # 5% initial uncertainty (variance)
Q_kf  = (0.002 * C_kf)**2     # process noise: allows E to drift ~0.2%/cycle
R_kf  = MEAS_NOISE_STRAIN**2  # measurement-noise variance

E_kf_hist  = np.zeros(n_cycles)
E_kf_std   = np.zeros(n_cycles)
innov_hist = np.zeros(n_cycles)

for k in range(n_cycles):
    H = alpha * F_cyc[k]                       # observation gain for this cycle
    C_kf, P_kf, innov = kalman_scalar_update(C_kf, P_kf, eps_cyc[k], H, Q_kf, R_kf)
    E_kf_hist[k]  = 1.0 / C_kf / 1e9           # posterior E estimate (GPa)
    E_kf_std[k]   = np.sqrt(P_kf) / C_kf**2 / 1e9  # σ_E from σ_C via E=1/C  (GPa)
    innov_hist[k] = innov * 1e6                # innovation in µε

cycle = np.arange(1, n_cycles+1)
E_true_gpa = E_true / 1e9
E_kf_2sig  = 2 * E_kf_std

fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
fig.suptitle('Digital Twin — Stage 3: Kalman Filter Tracking Fatigue Damage', fontsize=12)

# E estimate vs truth
ax = axes[0]
ax.plot(cycle, E_true_gpa,  'k--',  lw=1.2, alpha=0.5, label='True E')
ax.plot(cycle, E_kf_hist,   color='steelblue', lw=1.8, label='KF estimate')
ax.fill_between(cycle,
                E_kf_hist - E_kf_2sig,
                E_kf_hist + E_kf_2sig,
                alpha=0.25, color='steelblue', label='±2σ')
ax.axvline(damage_at, color='tomato', lw=1.5, ls=':', label=f'Damage event (cycle {damage_at})')
ax.set_ylabel('Estimated E (GPa)')
ax.set_title('Stiffness tracking — twin follows the damage event')
ax.legend(fontsize=9, ncol=2); ax.grid(True, linestyle=':', alpha=0.4)

# Innovation (measurement residual) — spike at damage onset
ax = axes[1]
win = 20
rolling_innov = np.convolve(np.abs(innov_hist), np.ones(win)/win, mode='same')
ax.plot(cycle, innov_hist, color='gray', lw=0.5, alpha=0.5)
ax.plot(cycle, rolling_innov, color='tomato', lw=2,
        label=f'Rolling |innovation| (w={win})')
ax.axvline(damage_at, color='tomato', lw=1.5, ls=':')
ax.axhline(0, color='black', lw=0.7)
ax.set_ylabel('Innovation (µε)')
ax.set_title('Measurement residual — spike flags the damage onset')
ax.legend(fontsize=9); ax.grid(True, linestyle=':', alpha=0.4)

# Tracking error
ax = axes[2]
error = np.abs(E_kf_hist - E_true_gpa)
ax.plot(cycle, error, color='seagreen', lw=1.5, label='|KF error| (GPa)')
ax.axvline(damage_at, color='tomato', lw=1.5, ls=':', label='Damage event')
ax.set_xlabel('Load cycle')
ax.set_ylabel('|E error| (GPa)')
ax.set_title('Absolute tracking error — recovers within ~100 cycles after damage')
ax.legend(fontsize=9); ax.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig62_nb1_kalman_damage_tracking.png', bbox_inches='tight', dpi=150)
plt.show()

# Detection: first cycle *after* the damage event where the estimate crosses
# below the midpoint between the healthy and damaged stiffness.
mid          = (E_pre + E_post) / 2e9          # GPa
detect_mask  = (cycle > damage_at) & (E_kf_hist < mid)
detect_cycle = int(cycle[detect_mask][0]) if detect_mask.any() else None

print(f"True damage at cycle:     {damage_at}")
if detect_cycle is not None:
    print(f"KF detects change at:     {detect_cycle}  (+{detect_cycle - damage_at} cycles lag)")
else:
    print(f"KF detects change at:     not detected within {n_cycles} cycles")
print(f"E before damage:  {E_kf_hist[:damage_at].mean():.2f} GPa  (true {E_pre/1e9:.0f} GPa)")
print(f"E after recovery: {E_kf_hist[-200:].mean():.2f} GPa  (true {E_post/1e9:.0f} GPa)")

---
## Where to take this next

This notebook tracks a single parameter with a scalar filter. Real digital twins are richer — and the chapter's cautions about observability, identifiability, and sensor fusion become concrete the moment you extend it. Three directions to build on:

1. **Estimate two parameters at once, and watch identifiability break.** Promote the state to a vector — say effective stiffness *and* an unknown tip load — and run a 2×2 Kalman filter. With a single gage the two trade off: a stiffness drop and a load increase can produce the *same* strain. Plot the posterior covariance and you will see the correlation the chapter's *identifiability* discussion warns about, then add a second gage at a different station and watch it resolve.

2. **Replace the abrupt step with gradual crack growth.** Swap the sudden 193 → 178 GPa step for a smooth decay (e.g. a Paris-law-style curve) and sweep the process noise *Q*. Measure detection lag versus estimate noise across *Q* values to map the tracking-speed/smoothness tradeoff the notebook only sketches — the classic Kalman tuning problem.

3. **Fuse two channels and inject a sensor fault.** Turn the measurement into a vector of two gages (observation matrix **H**, a 2×2 noise covariance **R**), then corrupt one channel with slow drift. Confirm you can see the *isolated single-channel residual* signature that Table 62.2 lists as the fingerprint of a sensor fault, as opposed to the spatially consistent pattern of true structural change.